# Browser observability 및 session replay

## 개요

이 튜토리얼에서는 AgentCore Browser 세션에 observability를 추가하고, 브라우저 console log와 network log, 전송된 CDP 명령, 브라우저에서 수행된 Agent 액션을 확인하는 방법을 알아봅니다.


### 튜토리얼 세부 정보


| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                           |
| Agent 유형          | 단일                                                                             |
| Agentic Framework   | Nova Act                                                                         |
| LLM 모델            | Amazon Nova Act model                                                            |
| 튜토리얼 구성 요소  | AgentCore Browser console에서 브라우저 log 관찰                                  |
| 튜토리얼 분야       | 범용                                                                             |
| 예제 난이도         | 쉬움                                                                             |
| 사용 SDK            | Amazon Bedrock AgentCore Python SDK, boto3 SDK, Nova Act                          |

### 튜토리얼 아키텍처

이 튜토리얼에서는 브라우저 console log, network log, CDP 명령 및 브라우저에서 수행된 Agent 액션을 관찰하는 방법을 단계별로 살펴봅니다.  


### 튜토리얼 주요 기능

* Browser tool의 세션 녹화 활성화
* Nova Act와 Browser tool을 함께 사용
* Log와 session replay 관찰

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.
* Python 3.10+
* AWS 자격 증명
* Amazon Bedrock AgentCore SDK
* Amazon Boto3 SDK
* Nova Act SDK 및 API 키 - https://nova.amazon.com/act 에서 API 키 생성

In [ ]:
!pip install  -r requirements.txt --quiet

## 녹화가 활성화된 사용자 지정 AgentCore Browser 리소스 생성
기본 Browser tool에는 녹화가 활성화되어 있지 않으므로 먼저 녹화가 활성화된 Browser tool 리소스를 생성해야 합니다. 그런 다음 이 브라우저 리소스를 사용해 브라우저 세션을 시작합니다.

In [ ]:
## 브라우저 녹화를 저장할 S3 bucket 생성
## 기존 bucket을 사용하려면 이 단계를 건너뛰고 다음 단계에서 bucket 이름을 수정하세요.
import boto3
import uuid
from boto3.session import Session

boto_session = Session()

region = boto_session.region_name
s3_client = boto3.client("s3", region_name=region)

bucket_name = f"agentcore-browser-recordings-{str(uuid.uuid4())[:8]}"
s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={"LocationConstraint": region})
print(f"Created S3 bucket: {bucket_name}")

### Browser tool 리소스 생성에 필요한 권한이 있는 execution role 생성

올바른 권한이 있는 execution role을 생성하는 helper utility를 만들어 보겠습니다.

In [ ]:
## 브라우저 생성 권한이 있는 execution role 생성
def create_agentcore_role(agent_name):
    iam_client = boto3.client("iam")
    agentcore_role_name = f"agentcore-{agent_name}-role"
    boto_session = Session()
    region = boto_session.region_name
    account_id = boto3.client("sts").get_caller_identity()["Account"]

    role_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "BrowserPermissions",
                "Effect": "Allow",
                "Action": [
                    "bedrock-agentcore:ConnectBrowserAutomationStream",
                    "bedrock-agentcore:ListBrowsers",
                    "bedrock-agentcore:GetBrowserSession",
                    "bedrock-agentcore:ListBrowserSessions",
                    "bedrock-agentcore:CreateBrowser",
                    "bedrock-agentcore:StartBrowserSession",
                    "bedrock-agentcore:StopBrowserSession",
                    "bedrock-agentcore:ConnectBrowserLiveViewStream",
                    "bedrock-agentcore:UpdateBrowserStream",
                    "bedrock-agentcore:DeleteBrowser",
                    "bedrock-agentcore:GetBrowser",
                ],
                "Resource": "*",
            },
            {
                "Sid": "S3Permissions",
                "Effect": "Allow",
                "Action": ["s3:PutObject", "s3:GetObject", "s3:ListBucket"],
                "Resource": [
                    f"arn:aws:s3:::{bucket_name}",
                    f"arn:aws:s3:::{bucket_name}/*",
                ],
            },
            {
                "Sid": "CloudWatchLogsPermissions",
                "Effect": "Allow",
                "Action": [
                    "logs:CreateLogGroup",
                    "logs:CreateLogStream",
                    "logs:PutLogEvents",
                    "logs:DescribeLogStreams",
                ],
                "Resource": "*",
            },
        ],
    }
    assume_role_policy_document = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "AssumeRolePolicy",
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
                "Condition": {
                    "StringEquals": {"aws:SourceAccount": f"{account_id}"},
                    "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{region}:{account_id}:*"},
                },
            }
        ],
    }

    assume_role_policy_document_json = json.dumps(assume_role_policy_document)
    role_policy_document = json.dumps(role_policy)

    try:
        # Trust policy를 사용해 IAM Role 생성
        agentcore_iam_role = iam_client.create_role(
            RoleName=agentcore_role_name,
            AssumeRolePolicyDocument=assume_role_policy_document_json,
        )
        print(f"Role {agentcore_role_name} created successfully.")

        # Role에 inline permissions policy 연결
        iam_client.put_role_policy(
            RoleName=agentcore_role_name,
            PolicyName=f"{agentcore_role_name}-inline-policy",
            PolicyDocument=role_policy_document,
        )
        print(f"Inline policy attached to role {agentcore_role_name}.")

    except iam_client.exceptions.EntityAlreadyExistsException:
        print("Role already exists -- deleting and creating it again")

        # 기존 inline policy 연결 해제 및 삭제
        policies = iam_client.list_role_policies(RoleName=agentcore_role_name)
        for policy_name in policies["PolicyNames"]:
            iam_client.delete_role_policy(RoleName=agentcore_role_name, PolicyName=policy_name)

        # Role 삭제 후 다시 생성
        print(f"Deleting role {agentcore_role_name}...")
        iam_client.delete_role(RoleName=agentcore_role_name)
        print(f"Recreating role {agentcore_role_name}...")

        agentcore_iam_role = iam_client.create_role(
            RoleName=agentcore_role_name,
            AssumeRolePolicyDocument=assume_role_policy_document_json,
        )
        print(f"Role {agentcore_role_name} recreated successfully.")

        # 다시 생성한 role에 inline permissions policy 재연결
        iam_client.put_role_policy(
            RoleName=agentcore_role_name,
            PolicyName=f"{agentcore_role_name}-inline-policy",
            PolicyDocument=role_policy_document,
        )
        print(f"Inline policy re-attached to role {agentcore_role_name}.")

    # 변경 사항이 전파되도록 잠시 대기
    time.sleep(10)

    return agentcore_iam_role

### 녹화가 활성화된 Browser tool 리소스 생성

In [ ]:
## boto3로 녹화가 활성화된 사용자 지정 Browser tool 리소스 생성
import boto3
import time
import json
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

cp_client = boto3.client("bedrock-agentcore-control", region_name=region)

# Browser Tool parameter 정의
browser_name = "my_custom_browser"
browser_description = "Test browser for observability and session replay"
execution_role = create_agentcore_role(browser_name)
execution_role_arn = execution_role["Role"]["Arn"]  # TODO: 사용할 IAM role ARN으로 교체
s3_bucket_name = bucket_name  # 기존 bucket이 있으면 해당 S3 bucket 이름으로 교체
s3_prefix = "replay-data"

try:
    response = cp_client.create_browser(
        name=browser_name,
        description=browser_description,
        networkConfiguration={
            "networkMode": "PUBLIC"  # VPC 통합이 필요하면 "VPC" 사용
        },
        executionRoleArn=execution_role_arn,
        clientToken=str(uuid.uuid4()),  # 멱등성을 위한 고유 token
        recording={
            "enabled": True,
            "s3Location": {"bucket": s3_bucket_name, "prefix": s3_prefix},
        },
    )
    print(response)
    print(f"Successfully created Browser Tool: {response['browserId']}")
    browserId = response["browserId"]
except cp_client.exceptions.ConflictException:
    print("Browser Tool with this name already exists. Please choose a different name.")

## Nova Act 스크립트 생성
Nova Act는 이전 단계에서 생성한 Browser tool 리소스를 사용해 브라우저 세션을 시작하고 브라우저 액션을 실행합니다.

In [ ]:
%%writefile basic_browser_with_nova_act.py
"""Amazon Bedrock AgentCore와 Nova Act를 사용하는 브라우저 자동화 스크립트입니다.

이 스크립트는 다음과 같은 AI 기반 웹 자동화를 보여 줍니다:
- Amazon Bedrock AgentCore를 통한 브라우저 세션 초기화
- 자연어 웹 상호 작용을 위한 Nova Act 연결
- 브라우저를 사용한 자동 검색 및 데이터 추출
"""

from bedrock_agentcore.tools.browser_client import browser_session , BrowserClient
from nova_act import NovaAct
from rich.console import Console
import argparse
import json

console = Console()

from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print("using region", region)

def browser_with_nova_act(prompt, starting_page, nova_act_key,  browserId, region="us-west-2"):
    result = None
    
    browser_client = BrowserClient(region)
    browser_client.start(identifier=browserId) # 여기에서 생성한 Browser tool ID 사용
    
    ws_url, headers = browser_client.generate_ws_headers()
    try:
        with NovaAct(
            cdp_endpoint_url=ws_url,
            cdp_headers=headers,
            nova_act_api_key=nova_act_key,
            starting_page=starting_page,
        ) as nova_act:
            result = nova_act.act(prompt)
    except Exception as e:
        console.print(f"NovaAct error: {e}")

    finally:
        browser_client.stop()
        return result


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", required=True, help="Browser Search instruction")
    parser.add_argument("--starting-page", required=True, help="Starting URL")
    parser.add_argument("--nova-act-key", required=True, help="Nova Act API key")
    parser.add_argument("--region", default="us-west-2", help="AWS region")
    parser.add_argument("--browserID", required=True, help="Browser Tool ID to use")
    args = parser.parse_args()

    result = browser_with_nova_act(
        args.prompt, args.starting_page, args.nova_act_key, args.browserID, args.region
    )
    console.print(f"\n[cyan] Response[/cyan] {result.response}")
    console.print(f"\n[bold green]Nova Act Result:[/bold green] {result}")

#### 스크립트 실행하기
스크립트를 실행하기 전에 아래에 Nova Act API 키를 입력합니다.

In [ ]:
NOVA_ACT_KEY = ""  ### 여기에 Nova Act API 키를 입력하세요

In [ ]:
!python basic_browser_with_nova_act.py --prompt "Search for macbooks and extract the details of the first one" --starting-page "https://www.amazon.com/" --browserID {browserId} --nova-act-key {NOVA_ACT_KEY}

## AgentCore Browser Console의 observability
* 스크립트가 실행되는 동안 AWS Console(https://us-west-2.console.aws.amazon.com/bedrock-agentcore/builtInTools)로 이동해
"Browser use tools" 탭을 클릭합니다. 다른 리전에서 실행 중이라면 이 URL의 리전을 변경하세요.
* "my-custom-browser"를 클릭합니다. 세션이 아직 실행 중이면 Live View 링크가, 종료되었다면 녹화 보기 링크가 표시됩니다.
* 세션이 실행 중이면 종료될 때까지 기다린 다음 녹화 보기를 클릭합니다.
다음과 유사한 페이지가 표시됩니다.

![image](./images/browser_recording_1.png)

* #### 이제 녹화된 브라우저 세션을 재생할 수 있습니다.
* #### 세션 중 방문한 각 페이지를 살펴볼 수 있습니다.
* #### Action 탭에서 Agent가 수행한 액션을 확인할 수 있습니다.
* #### Page DOM 세부 정보, Console log, 브라우저로 전송된 CDP 명령 및 Network log를 확인할 수 있습니다.
* #### 추가 디버깅을 위해 각 log를 다운로드할 수 있습니다.
* #### Actions 탭에서 각 액션의 "View"를 클릭해 브라우저에서 수행된 정확한 액션을 확인할 수 있습니다.
* #### 예: "Click" 유형 액션 중 하나를 열고 브라우저의 빨간색 원을 확인하세요. 이 원은 클릭 액션이 발생한 정확한 위치를 나타냅니다.

# 축하합니다. 즐겁게 살펴보세요!